# 01 — Exploratory Data Analysis

**Goal:** understand how real photos and AI-generated images differ before training any model.

The rubric requires at least three image-property comparisons across real images and the AI sources. This notebook covers class balance, brightness, sharpness, and a side-by-side caption-family comparison.

## Setup

Import libraries and define the source labels we'll use throughout the project.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from datasets import load_dataset
import cv2
from tqdm import tqdm

OUTPUT_DIR = "../Outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Map Label_B integers to readable model names.
# We use only three sources in this case study: Real, SD3, DALL·E 3.
MODEL_NAMES = {
    0: "Real",
    3: "SD3",
    4: "DALL·E 3",
}

## Load the dataset

We pull the first 1,000 rows from Hugging Face for a quick EDA. For a full analysis you would use the entire dataset.

In [ ]:
print("Loading dataset (first 1000 rows for quick EDA)...")
ds = load_dataset("Rajarshi-Roy-research/Defactify_Image_Dataset",
                  split="train[:1000]")
ds = ds.filter(lambda x: x["Label_B"] in [0, 3, 4])

df = pd.DataFrame({
    "Label_A": ds["Label_A"],
    "Label_B": ds["Label_B"],
    "Caption": ds["Caption"],
})
df["Model"] = df["Label_B"].map(MODEL_NAMES)
print(f"Loaded {len(df)} rows after filtering.")
df.head()

## Plot 1 — Class balance (Real vs. AI-generated)

The dataset is heavily imbalanced toward AI images. A model that always predicts "AI" will look deceptively accurate, so we'll need to track F1 and ROC-AUC alongside accuracy.

In [ ]:
counts = df["Label_A"].map({0: "Real", 1: "AI-generated"}).value_counts()
plt.figure(figsize=(7, 4))
plt.bar(counts.index, counts.values, color=["#2ecc71", "#e74c3c"])
plt.title("Class Balance: Real vs. AI-Generated")
plt.ylabel("Number of Images")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/01_label_a_distribution.png", dpi=120)
plt.show()

## Plot 2 — Per-source distribution

How many examples do we have from each generator?

In [ ]:
src_counts = df["Model"].value_counts()
plt.figure(figsize=(7, 4))
plt.bar(src_counts.index, src_counts.values,
        color=["#2ecc71", "#9b59b6", "#3498db"])
plt.title("Image Counts by Source (Label_B)")
plt.ylabel("Number of Images")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/02_label_b_distribution.png", dpi=120)
plt.show()

## Plot 3 — Brightness comparison

Compute mean grayscale brightness for every image and box-plot the distributions per source. Different generators sometimes produce systematically brighter or darker images.

In [ ]:
brightness = {label: [] for label in MODEL_NAMES.values()}

for ex in tqdm(ds, desc="Computing brightness"):
    img = ex["Image"]
    if not isinstance(img, Image.Image):
        img = Image.fromarray(img)
    arr = np.array(img.convert("L"))   # grayscale
    brightness[MODEL_NAMES[ex["Label_B"]]].append(arr.mean())

plt.figure(figsize=(8, 5))
plt.boxplot(brightness.values(), labels=brightness.keys())
plt.title("Brightness Distribution by Source")
plt.ylabel("Mean Pixel Brightness (0–255)")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/03_brightness.png", dpi=120)
plt.show()

## Plot 4 — Sharpness (Laplacian variance)

Laplacian variance is a standard sharpness metric: high variance means crisp edges, low variance means smooth or blurry. AI-generated images often have characteristically smooth textures.

In [ ]:
sharpness = {label: [] for label in MODEL_NAMES.values()}

for ex in tqdm(ds, desc="Computing Laplacian variance"):
    img = ex["Image"]
    if not isinstance(img, Image.Image):
        img = Image.fromarray(img)
    gray = np.array(img.convert("L"))
    lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    sharpness[MODEL_NAMES[ex["Label_B"]]].append(lap_var)

plt.figure(figsize=(8, 5))
plt.boxplot(sharpness.values(), labels=sharpness.keys())
plt.title("Sharpness (Laplacian Variance) by Source")
plt.ylabel("Laplacian Variance")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/04_sharpness.png", dpi=120)
plt.show()

## Plot 5 — One real image side-by-side with its AI siblings

The Defactify dataset is organized into "caption families": one real image and several AI-generated images that share the same caption. This is a powerful way to see what each generator produces from the same prompt.

In [ ]:
caption_counts = df["Caption"].value_counts()
complete_captions = caption_counts[caption_counts >= 3].index

if len(complete_captions) > 0:
    sample_caption = complete_captions[0]
    family = [ex for ex in ds if ex["Caption"] == sample_caption]
    family.sort(key=lambda x: x["Label_B"])

    fig, axes = plt.subplots(1, len(family), figsize=(4 * len(family), 4))
    if len(family) == 1:
        axes = [axes]
    for ax, ex in zip(axes, family):
        ax.imshow(ex["Image"])
        ax.set_title(MODEL_NAMES[ex["Label_B"]])
        ax.axis("off")
    plt.suptitle(f"Caption: {sample_caption[:60]}...", fontsize=10)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/05_family_comparison.png", dpi=120)
    plt.show()

## Next step

EDA is done. Plots are saved to `../Outputs/`. Move on to **`02_train_CNN.ipynb`** to train your first model.